In [1]:
!pip install -q -U groq

import json
import re
from groq import Groq
from getpass import getpass

# ==========================================
# 1. API KEY
# ==========================================

api_key = getpass("Enter your Groq API Key: ")

client = Groq(api_key=api_key)


# ==========================================
# 2. MODEL CONFIGURATION
# ==========================================

GENERATOR_MODEL = "openai/gpt-oss-120b"

EVALUATOR_MODEL = "openai/gpt-oss-20b"


# ==========================================
# 3. LLM 1 - GENERATOR
# ==========================================

def generate_answer(question):

    response = client.chat.completions.create(

        model=GENERATOR_MODEL,

        messages=[
            {
                "role": "system",
                "content": """
                You are a professional AI assistant.
                Provide accurate, relevant, informative
                and clearly structured answers.
                """
            },
            {
                "role": "user",
                "content": question
            }
        ],

        temperature=0.3,
        max_completion_tokens=4000
    )

    answer = response.choices[0].message.content

    if not answer:
        raise ValueError("Generator returned an empty response.")

    return answer


# ==========================================
# 4. JSON EXTRACTION
# ==========================================

def extract_json(result):

    if not result or not result.strip():
        raise ValueError("Evaluator returned an empty response.")

    # Remove markdown code blocks

    result = re.sub(
        r"```(?:json)?",
        "",
        result,
        flags=re.IGNORECASE
    )

    result = result.replace("```", "").strip()

    # Extract JSON object

    match = re.search(r"\{.*\}", result, re.DOTALL)

    if not match:
        raise ValueError(
            "No valid JSON object found in evaluator response."
        )

    return json.loads(match.group())


# ==========================================
# 5. LLM 2 - STRICT EVALUATOR
# ==========================================

def evaluate_answer(question, generated_answer, reference_answer=None):

    reference = (
        reference_answer
        if reference_answer
        else "No reference answer provided. Evaluate using established factual knowledge."
    )

    evaluation_prompt = f"""

    You are a STRICT and INDEPENDENT AI evaluator.

    Evaluate the answer generated by another AI model.

    USER QUESTION:
    {question}

    REFERENCE ANSWER:
    {reference}

    GENERATED ANSWER:
    {generated_answer}

    ==========================================

    EVALUATION CRITERIA:

    1. CORRECTNESS (40 POINTS)

       - Check factual accuracy.
       - Identify incorrect information.
       - Identify misleading claims.
       - Penalize unsupported claims.

    2. RELEVANCE (25 POINTS)

       - Check whether the answer addresses
         the actual question.
       - Penalize irrelevant information.

    3. COMPLETENESS (20 POINTS)

       - Check missing important information.
       - Identify incomplete explanations.

    4. CLARITY (15 POINTS)

       - Check explanation quality.
       - Check readability.
       - Check logical organization.

    ==========================================

    STRICT RULES:

    1. Never automatically award 100%.
    2. Do not reward lengthy answers automatically.
    3. Evaluate factual accuracy carefully.
    4. Identify genuine weaknesses.
    5. Do not invent mistakes.
    6. Compare with the reference answer if available.
    7. Without a reference, acknowledge uncertainty.
    8. Award full marks only when justified.
    9. Scores must be integers.
    10. Total must equal the sum of all criteria.

    ==========================================

    RETURN JSON WITH EXACTLY THIS STRUCTURE:

    {{
        "correctness": 0,
        "relevance": 0,
        "completeness": 0,
        "clarity": 0,
        "total_score": 0,
        "weaknesses": [
            "Identified weakness"
        ],
        "feedback": "Detailed evaluation"
    }}

    """

    # ======================================
    # CALL EVALUATOR
    # ======================================

    response = client.chat.completions.create(

        model=EVALUATOR_MODEL,

        messages=[
            {
                "role": "system",
                "content": """
                You are a strict academic evaluator.
                Return only a valid JSON object.
                Evaluate objectively and never inflate scores.
                """
            },
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ],

        temperature=0,

        max_completion_tokens=3000,

        response_format={
            "type": "json_object"
        }
    )

    # ======================================
    # EXTRACT RESPONSE
    # ======================================

    result = response.choices[0].message.content

    print("\nEvaluator response received.")

    evaluation = extract_json(result)

    # ======================================
    # VALIDATE SCORES
    # ======================================

    score_limits = {

        "correctness": 40,
        "relevance": 25,
        "completeness": 20,
        "clarity": 15

    }

    for criterion, maximum in score_limits.items():

        score = evaluation.get(criterion)

        if type(score) != int or not 0 <= score <= maximum:

            raise ValueError(
                f"Invalid score for {criterion}: {score}"
            )

    calculated_total = sum(

        evaluation[key]

        for key in score_limits

    )

    if calculated_total != evaluation.get("total_score"):

        raise ValueError(
            "Evaluator score calculation mismatch!"
        )

    if not isinstance(evaluation.get("weaknesses"), list):

        raise ValueError("Invalid weaknesses format.")

    # ======================================
    # DETERMINE STATUS
    # ======================================

    if calculated_total >= 90:

        status = "EXCELLENT"

    elif calculated_total >= 80:

        status = "GOOD"

    elif calculated_total >= 70:

        status = "AVERAGE"

    elif calculated_total >= 50:

        status = "NEEDS IMPROVEMENT"

    else:

        status = "POOR"

    evaluation["status"] = status

    return evaluation


# ==========================================
# 6. MAIN EXECUTION
# ==========================================

print("=" * 55)

print("          TWO-LLM EVALUATION SYSTEM")

print("=" * 55)

question = input("\nEnter your question: ")

reference_answer = input(
    "\nEnter reference answer (optional, press Enter to skip): "
)

if not question.strip():

    raise ValueError("Question cannot be empty!")


try:

    # ======================================
    # GENERATOR
    # ======================================

    print("\nGenerating answer using LLM 1...\n")

    generated_answer = generate_answer(question)

    print("\n========== GENERATOR OUTPUT ==========\n")

    print(generated_answer)


    # ======================================
    # EVALUATOR
    # ======================================

    print("\nEvaluating answer using LLM 2...\n")

    evaluation = evaluate_answer(

        question,

        generated_answer,

        reference_answer if reference_answer.strip() else None

    )


    # ======================================
    # FINAL REPORT
    # ======================================

    print("\n")

    print("=" * 55)

    print("             EVALUATION REPORT")

    print("=" * 55)

    print(
        f"\nCorrectness    : {evaluation['correctness']} / 40"
    )

    print(
        f"Relevance      : {evaluation['relevance']} / 25"
    )

    print(
        f"Completeness   : {evaluation['completeness']} / 20"
    )

    print(
        f"Clarity        : {evaluation['clarity']} / 15"
    )

    print("-" * 55)

    print(
        f"FINAL SCORE    : {evaluation['total_score']}%"
    )

    print(
        f"STATUS         : {evaluation['status']}"
    )

    print("\nWEAKNESSES:")

    if evaluation["weaknesses"]:

        for i, weakness in enumerate(
            evaluation["weaknesses"], 1
        ):

            print(f"{i}. {weakness}")

    else:

        print("No significant weaknesses identified.")

    print("\nFEEDBACK:")

    print(evaluation["feedback"])

    print("=" * 55)

    print("       EVALUATION COMPLETED")

    print("=" * 55)


except Exception as e:

    print("\n========== ERROR ==========")

    print("Evaluation failed:", str(e))

    print("\nPlease check the API response and try again.")

    print("===========================")

Enter your Groq API Key: ··········
          TWO-LLM EVALUATION SYSTEM

Enter your question: what is ai

Enter reference answer (optional, press Enter to skip): 

Generating answer using LLM 1...


========== GENERATOR OUTPUT ==========

**Artificial Intelligence (AI): A Concise Overview**

---

## 1. Definition
**Artificial Intelligence (AI)** is the field of computer science that aims to create machines or software capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem‑solving, perception, language understanding, and decision‑making.

> **In short:** AI = machines that can *think* or *act* like humans (to varying degrees).

---

## 2. Core Concepts

| Concept | What It Means | Typical Techniques |
|---------|---------------|--------------------|
| **Machine Learning (ML)** | Systems improve performance from data without being explicitly programmed. | Supervised/unsupervised learning, reinforcement learning, deep learnin